In [1]:
import os
import glob
import subprocess
import pandas as pd

# --- STEP 0: Paths and Folders ---
os.makedirs("data/raw", exist_ok=True)
os.makedirs("data/processed", exist_ok=True)

# Please note the gitignore is activated so that the data/raw/kpss_ext directory is not tracked by git. This is to avoid committing large datasets to the repository.

# --- STEP 1: Download KPSS ---
# Clone the KPSS extended dataset (updated through 2025) into data/raw/kpss_ext
repo_url = "https://github.com/KPSS2017/Technological-Innovation-Resource-Allocation-and-Growth-Extended-Data.git"
kpss_dir = "data/raw/kpss_ext"

if not os.path.exists(kpss_dir):
    subprocess.run(["git", "clone", "--depth", "1", repo_url, kpss_dir], check=True)

def load_kpss_table(prefix):
    matches = glob.glob(os.path.join(kpss_dir, f"{prefix}*"))
    if not matches:
        raise FileNotFoundError(f"Could not find file starting with {prefix} in {kpss_dir}")
    print(f"Loading: {matches[0]}")
    return pd.read_csv(matches[0], low_memory=False)

kpss = load_kpss_table("KPSS_2025")
firm = load_kpss_table("Match_patent_permco_permno_2025")
cpc  = load_kpss_table("Match_patent_cpc_2025")

print("\nShapes:", {"kpss": kpss.shape, "firm": firm.shape, "cpc": cpc.shape})
print("KPSS columns:", list(kpss.columns))
print("Firm columns:", list(firm.columns))
print("CPC columns:", list(cpc.columns))
print(kpss.head(3))



Loading: data/raw/kpss_ext\KPSS_2025.zip
Loading: data/raw/kpss_ext\Match_patent_permco_permno_2025.zip
Loading: data/raw/kpss_ext\Match_patent_cpc_2025.zip

Shapes: {'kpss': (3831908, 7), 'firm': (3831908, 3), 'cpc': (3831908, 2)}
KPSS columns: ['patent_num', 'permno', 'issue_date', 'filing_date', 'xi_nominal', 'xi_real', 'cites']
Firm columns: ['patent_num', 'permco', 'permno']
CPC columns: ['patent_num', 'cpc']
   patent_num  permno  issue_date  filing_date  xi_nominal   xi_real  cites
0     1571164   10751    19260126   19250701.0    0.037976  0.209402      1
1     1570992   10786    19260126   19231221.0    0.030295  0.167048      0
2     1571010   10401    19260126   19201223.0    0.214299  1.181669      8


In [7]:
# --- STEP 2: Verify KPSS and manual USPTO downloads ---

from pathlib import Path

# A notebook normally starts with the notebook directory as its working directory,
# so prefer it first and fall back to the repository directory.
cwd = Path.cwd()
raw_candidates = [cwd / "data" / "raw", cwd.parent / "data" / "raw"]
raw_root = next((p for p in raw_candidates if p.exists()), cwd)

kpss_matches = sorted(raw_root.rglob("*KPSS_2025*")) + sorted(raw_root.rglob("*Match_patent_*2025*"))

# Match the USPTO directory by name, regardless of how it is nested.
uspto_dirs = [p for p in raw_root.rglob("*") if p.is_dir() and p.name.lower() == "uspto"]

print("=== STEP 2 CHECK ===")
print(f"Working directory: {cwd}")
print(f"Raw-data root: {raw_root}")
print(f"KPSS files found: {len(kpss_matches)}")
for f in kpss_matches:
    print(f"  [KPSS]  {f.relative_to(raw_root)} ({f.stat().st_size / (1024*1024):.1f} MB)")

uspto_files = []
for udir in uspto_dirs:
    for f in sorted(udir.rglob("*")):
        if f.is_file():
            uspto_files.append(f)
            print(f"  [USPTO] {f.relative_to(raw_root)} ({f.stat().st_size / (1024*1024):.1f} MB)")

print(f"USPTO directories found: {len(uspto_dirs)}")
print(f"USPTO files found: {len(uspto_files)}")
if not uspto_dirs:
    print("  [USPTO] No 'uspto' folder found below:", raw_root)


=== STEP 2 CHECK ===
Working directory: c:\Users\justy\OneDrive\Pulpit\MBA\Master Thesis\Thesis_Folder\digital-patents-event-study\notebooks
Raw-data root: c:\Users\justy\OneDrive\Pulpit\MBA\Master Thesis\Thesis_Folder\digital-patents-event-study\notebooks\data\raw
KPSS files found: 3
  [KPSS]  kpss_ext\KPSS_2025.zip (59.6 MB)
  [KPSS]  kpss_ext\Match_patent_cpc_2025.zip (76.3 MB)
  [KPSS]  kpss_ext\Match_patent_permco_permno_2025.zip (17.6 MB)
  [USPTO] uspto\g_draw_desc_text_2025.tsv.zip (132.3 MB)
  [USPTO] uspto\g_uspc_at_issue.tsv.zip (585.2 MB)
USPTO directories found: 1
USPTO files found: 2


In [12]:
# --- STEP 3: Inspect KPSS and USPTO ZIP tables without writing extracted files ---
from pathlib import Path
import zipfile
import pandas as pd

# Locate the repository root and raw data.
cwd = Path.cwd()
root = next((p for p in [cwd, *cwd.parents] if (p / ".git").exists()), cwd)

raw_dir = root / "notebooks" / "data" / "raw"
if not raw_dir.exists():
    raw_dir = root / "data" / "raw"

processed_dir = raw_dir.parent / "processed"
processed_dir.mkdir(parents=True, exist_ok=True)

print("Repository root:", root)
print("Raw-data directory:", raw_dir)


def find_zip(filename_pattern):
    matches = sorted(raw_dir.rglob(filename_pattern))
    if not matches:
        raise FileNotFoundError(
            f"Missing raw ZIP matching '{filename_pattern}' below {raw_dir}"
        )
    return matches[0]


def read_zip_table(zip_pattern, member_pattern=None, sep=","):
    """Read a single member directly from a ZIP archive."""
    zpath = find_zip(zip_pattern)
    print(f"Reading archive: {zpath.relative_to(root)}")

    with zipfile.ZipFile(zpath, "r") as zf:
        members = [
            name for name in zf.namelist()
            if not name.endswith("/")
            and (member_pattern is None or Path(name).match(member_pattern))
        ]

    if len(members) != 1:
        raise RuntimeError(
            f"Expected one matching member in {zpath.name}, found {len(members)}: {members[:10]}"
        )

    with zipfile.ZipFile(zpath, "r") as zf:
        with zf.open(members[0]) as member:
            return pd.read_csv(member, sep=sep, low_memory=False)


kpss = read_zip_table("*KPSS_2025*.zip")
firm = read_zip_table("*Match_patent_permco_permno_2025*.zip")
cpc = read_zip_table("*Match_patent_cpc_2025*.zip")

# USPTO has two separate TSV ZIP archives.
uspc = read_zip_table("g_uspc_at_issue*.zip", sep="\t")
drawings = read_zip_table("g_draw_desc_text*.zip", sep="\t")

for df in (kpss, firm, cpc, uspc, drawings):
    df.columns = [c.strip().lower() for c in df.columns]

print("\nShapes:", {
    "kpss": kpss.shape,
    "firm": firm.shape,
    "cpc": cpc.shape,
    "uspc": uspc.shape,
    "drawings": drawings.shape,
})
print("KPSS cols:", list(kpss.columns))
print("Firm cols:", list(firm.columns))
print("CPC cols:", list(cpc.columns))
print("USPC cols:", list(uspc.columns))
print("USPTO drawings/description cols:", list(drawings.columns))
print("\nUSPC sample:\n", uspc.head(3))
print("\nDrawing/description sample:\n", drawings.head(3))

Repository root: c:\Users\justy\OneDrive\Pulpit\MBA\Master Thesis\Thesis_Folder\digital-patents-event-study
Raw-data directory: c:\Users\justy\OneDrive\Pulpit\MBA\Master Thesis\Thesis_Folder\digital-patents-event-study\notebooks\data\raw
Reading archive: notebooks\data\raw\kpss_ext\KPSS_2025.zip
Reading archive: notebooks\data\raw\kpss_ext\Match_patent_permco_permno_2025.zip
Reading archive: notebooks\data\raw\kpss_ext\Match_patent_cpc_2025.zip
Reading archive: notebooks\data\raw\uspto\g_uspc_at_issue.tsv.zip
Reading archive: notebooks\data\raw\uspto\g_draw_desc_text_2025.tsv.zip

Shapes: {'kpss': (3831908, 7), 'firm': (3831908, 3), 'cpc': (3831908, 2), 'uspc': (20953120, 6), 'drawings': (5373121, 3)}
KPSS cols: ['patent_num', 'permno', 'issue_date', 'filing_date', 'xi_nominal', 'xi_real', 'cites']
Firm cols: ['patent_num', 'permco', 'permno']
CPC cols: ['patent_num', 'cpc']
USPC cols: ['patent_id', 'uspc_sequence', 'uspc_mainclass_id', 'uspc_mainclass_title', 'uspc_subclass_id', 'uspc

In [20]:
# --- STEP 4: Install DuckDB for Database ---

%pip install duckdb


  Using cached duckdb-2.0.1-cp312-abi3-win_amd64.whl.metadata (4.1 kB)
Using cached duckdb-2.0.1-cp312-abi3-win_amd64.whl (16.1 MB)
Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.2 -> 26.3
[notice] To update, run: python.exe -m pip install --upgrade pip


In [ ]:
# --- STEP 5: Create local DuckDB database and load the three raw KPSS tables ---
from pathlib import Path
import duckdb

cwd = Path.cwd()
root = next((p for p in [cwd, *cwd.parents] if p.name == "digital-patents-event-study"), cwd)
extracted_dir = next(root.rglob("extracted"), None)

if extracted_dir is None:
    raise FileNotFoundError("Could not find the manually extracted data directory.")

db_path = root / "data" / "patents_2010_2025.duckdb"
db_path.parent.mkdir(parents=True, exist_ok=True)

kpss_csv = next(extracted_dir.rglob("KPSS_2025.csv")).as_posix()
firm_csv = next(extracted_dir.rglob("Match_patent_permco_permno_2025.csv")).as_posix()
cpc_csv = next(extracted_dir.rglob("Match_patent_cpc_2025.csv")).as_posix()

con = duckdb.connect(str(db_path))

# Raw dimensions loaded as-is; they are retained for data investigation.
con.execute(f"CREATE OR REPLACE TABLE kpss_firm AS SELECT * FROM read_csv_auto('{firm_csv}')")
con.execute(f"CREATE OR REPLACE TABLE kpss_cpc AS SELECT * FROM read_csv_auto('{cpc_csv}')")

con.execute(f"""
CREATE OR REPLACE TABLE kpss_10_25 AS
WITH parsed AS (
    SELECT
        CAST(patent_num AS BIGINT) AS patent_num,
        CAST(permno AS BIGINT) AS permno,
        CAST(TRY_STRPTIME(CAST(issue_date AS VARCHAR), '%Y%m%d') AS DATE) AS issue_date,
        CAST(TRY_STRPTIME(CAST(filing_date AS VARCHAR), '%Y%m%d') AS DATE) AS filing_date,
        CAST(xi_nominal AS DOUBLE) AS xi_nominal,
        CAST(xi_real AS DOUBLE) AS xi_real,
        CAST(cites AS BIGINT) AS cites
    FROM read_csv_auto('{kpss_csv}')
)
SELECT *
FROM parsed
WHERE issue_date IS NOT NULL
  AND EXTRACT(YEAR FROM issue_date) BETWEEN 2010 AND 2025
""")

print("Database:", db_path)
print("kpss_10_25 row count:", con.execute("SELECT COUNT(*) FROM kpss_10_25").fetchone()[0])
print(con.execute("SELECT * FROM kpss_10_25 LIMIT 5").df())


kpss_10_25 row count: 1725791
   patent_num  permno issue_date filing_date  xi_nominal    xi_real  cites
0     7640654   53727 2010-01-05  2007-02-02    0.172616   0.076865      0
1     7640659   53727 2010-01-05  2005-08-11    0.172616   0.076865      1
2     7644445   10107 2010-01-05  2005-07-11   31.533810  14.041947      2
3     7644444   89335 2010-01-05  2003-10-14    0.324892   0.144674      4
4     7644407   59328 2010-01-05  2004-03-05   13.934193   6.204870      7


In [6]:
# --- STEP 6: Inspect KPSS_firm and KPSS_cpc tables ---

print("--- kpss_firm ---")
print(con.execute("DESCRIBE kpss_firm").df()[["column_name", "column_type"]])
print(con.execute("SELECT * FROM kpss_firm LIMIT 3").df())

print("\n--- kpss_cpc ---")
print(con.execute("DESCRIBE kpss_cpc").df()[["column_name", "column_type"]])
print(con.execute("SELECT * FROM kpss_cpc LIMIT 3").df())

print("\n--- Quick Match Counts on 2010-2025 ---")
print(con.execute("""
SELECT
    COUNT(*) AS kpss_10_25_rows,
    COUNT(DISTINCT patent_num) AS unique_patents,
    COUNT(DISTINCT permno) AS unique_permnos_in_kpss
FROM kpss_10_25
""").df())

--- kpss_firm ---
  column_name column_type
0  patent_num      BIGINT
1      permco      BIGINT
2      permno      BIGINT
   patent_num  permco  permno
0     1571164   22223   10751
1     1570992   20299   10786
2     1571010   20103   10401

--- kpss_cpc ---
  column_name column_type
0  patent_num      BIGINT
1         cpc     VARCHAR
   patent_num                              cpc
0     1570604                         F23K3/12
1     1570665  B26D3/003;B26D3/16;Y10T82/16229
2     1570677              F41A19/15;F41A19/49

--- Quick Match Counts on 2010-2025 ---
   kpss_10_25_rows  unique_patents  unique_permnos_in_kpss
0          1725791         1725791                    3890


In [7]:
# --- STEP 7: Investigate keys, data types, and constraints (no *_clean tables) ---

# Remove intermediate cleaned tables from earlier runs.
con.execute("DROP TABLE IF EXISTS firm_clean")
con.execute("DROP TABLE IF EXISTS cpc_clean")
con.execute("DROP TABLE IF EXISTS kpss_panel_10_25")

TABLES = {
    "kpss_10_25": ["patent_num"],
    "kpss_firm": ["patent_num", "permno", "permco"],
    "kpss_cpc": ["patent_num", "cpc"],
}

print("=== Dataset key investigation ===")
for table_name, candidate_columns in TABLES.items():
    row_count = con.execute(f"SELECT COUNT(*) FROM {table_name}").fetchone()[0]
    print(f"\n{table_name}: {row_count:,} rows")

    schema_df = con.execute(f"DESCRIBE {table_name}").df()
    print(schema_df[["column_name", "column_type", "null", "key", "default"]].to_string(index=False))

    for column in candidate_columns:
        stats = con.execute(f"""
        SELECT
            COUNT(*) AS non_null_rows,
            COUNT(DISTINCT {column}) AS distinct_values,
            COUNT(*) - COUNT({column}) AS null_values
        FROM {table_name}
        """).fetchone()

        duplicate_count = con.execute(f"""
        SELECT COUNT(*)
        FROM (
            SELECT {column}
            FROM {table_name}
            WHERE {column} IS NOT NULL
            GROUP BY {column}
            HAVING COUNT(*) > 1
        )
        """).fetchone()[0]

        print(
            f"  {column}: distinct={stats[1]:,}, "
            f"nulls={stats[2]:,}, duplicate_values={duplicate_count:,}"
        )

    if table_name == "kpss_firm":
        duplicate_rows = con.execute("""
        SELECT COUNT(*)
        FROM (
            SELECT patent_num, permno, permco
            FROM kpss_firm
            GROUP BY patent_num, permno, permco
            HAVING COUNT(*) > 1
        )
        """).fetchone()[0]
        print(f"  Composite (patent_num, permno, permco): duplicate_rows={duplicate_rows:,}")

    if table_name == "kpss_cpc":
        duplicate_rows = con.execute("""
        SELECT COUNT(*)
        FROM (
            SELECT patent_num, cpc
            FROM kpss_cpc
            GROUP BY patent_num, cpc
            HAVING COUNT(*) > 1
        )
        """).fetchone()[0]
        print(f"  Composite (patent_num, cpc): duplicate_rows={duplicate_rows:,}")

print("\n=== Primary-key candidates ===")
print("kpss_10_25: patent_num (verify duplicate_values is 0 before assignment)")
print("kpss_firm: patent_num + permno + permco composite, only if duplicate_rows is 0")
print("kpss_cpc: patent_num + cpc composite, only if duplicate_rows is 0")

print("\n=== Cross-table uniqueness checks ===")
print(con.execute("""
SELECT
    COUNT(*) AS kpss_rows,
    COUNT(DISTINCT patent_num) AS distinct_kpss_patents,
    COUNT(*) - COUNT(DISTINCT patent_num) AS duplicate_patent_nums
FROM kpss_10_25
""").df().to_string(index=False))

print(con.execute("""
SELECT
    COUNT(*) AS firm_rows,
    COUNT(DISTINCT patent_num) AS distinct_firm_patents,
    COUNT(DISTINCT concat(patent_num, '|', permno, '|', permco)) AS distinct_firm_combinations
FROM kpss_firm
""").df().to_string(index=False))

print(con.execute("""
SELECT
    COUNT(*) AS cpc_rows,
    COUNT(DISTINCT patent_num) AS distinct_cpc_patents,
    COUNT(DISTINCT concat(patent_num, '|', cpc)) AS distinct_cpc_combinations
FROM kpss_cpc
""").df().to_string(index=False))


=== Dataset key investigation ===

kpss_10_25: 1,725,791 rows
column_name column_type null  key default
 patent_num      BIGINT  YES None    None
     permno      BIGINT  YES None    None
 issue_date        DATE  YES None    None
filing_date        DATE  YES None    None
 xi_nominal      DOUBLE  YES None    None
    xi_real      DOUBLE  YES None    None
      cites      BIGINT  YES None    None
  patent_num: distinct=1,725,791, nulls=0, duplicate_values=0

kpss_firm: 3,831,908 rows
column_name column_type null  key default
 patent_num      BIGINT  YES None    None
     permco      BIGINT  YES None    None
     permno      BIGINT  YES None    None
  patent_num: distinct=3,831,908, nulls=0, duplicate_values=0
  permno: distinct=10,599, nulls=0, duplicate_values=8,892
  permco: distinct=10,468, nulls=0, duplicate_values=8,793
  Composite (patent_num, permno, permco): duplicate_rows=0

kpss_cpc: 3,831,908 rows
column_name column_type null  key default
 patent_num      BIGINT  YES None    N

In [8]:
# --- STEP 8: Investigate kpss_10_25 keys and date fields ---

print("=== kpss_10_25 basic profile ===")
profile = con.execute("""
SELECT
    COUNT(*) AS row_count,
    COUNT(DISTINCT patent_num) AS distinct_patents,
    COUNT(*) - COUNT(DISTINCT patent_num) AS duplicate_patent_rows,
    COUNT(*) - COUNT(patent_num) AS null_patent_num,
    COUNT(*) - COUNT(permno) AS null_permno,
    COUNT(*) - COUNT(issue_date) AS null_issue_date,
    COUNT(*) - COUNT(filing_date) AS null_filing_date,
    MIN(issue_date) AS min_issue_date,
    MAX(issue_date) AS max_issue_date,
    MIN(filing_date) AS min_filing_date,
    MAX(filing_date) AS max_filing_date
FROM kpss_10_25
""").df()
print(profile.to_string(index=False))

# Duplicate patent numbers and a few examples.
print("\n=== Duplicate patent numbers ===")
print(con.execute("""
SELECT
    patent_num,
    COUNT(*) AS occurrences,
    COUNT(DISTINCT permno) AS distinct_permnos
FROM kpss_10_25
GROUP BY patent_num
HAVING COUNT(*) > 1
ORDER BY occurrences DESC, patent_num
LIMIT 20
""").df())

# Issue-date distribution by year.
print("\n=== Issue-date distribution by year ===")
issue_years = con.execute("""
SELECT
    YEAR(issue_date) AS issue_year,
    COUNT(*) AS patent_rows,
    COUNT(DISTINCT patent_num) AS distinct_patents
FROM kpss_10_25
WHERE issue_date IS NOT NULL
GROUP BY issue_year
ORDER BY issue_year
""").df()
print(issue_years.to_string(index=False))

# Filing-date distribution by year.
print("\n=== Filing-date distribution by year ===")
filing_years = con.execute("""
SELECT
    YEAR(filing_date) AS filing_year,
    COUNT(*) AS patent_rows,
    COUNT(DISTINCT patent_num) AS distinct_patents
FROM kpss_10_25
WHERE filing_date IS NOT NULL
GROUP BY filing_year
ORDER BY filing_year
""").df()
print(filing_years.to_string(index=False))

# Date summary: range and quartiles for additional distribution detail.
print("\n=== Date statistics ===")
print(con.execute("""
SELECT
    'issue_date' AS field,
    COUNT(issue_date) AS non_null_count,
    MIN(issue_date) AS minimum,
    APPROX_QUANTILE(issue_date, 0.25) AS q1,
    APPROX_QUANTILE(issue_date, 0.50) AS median,
    APPROX_QUANTILE(issue_date, 0.75) AS q3,
    MAX(issue_date) AS maximum
FROM kpss_10_25
UNION ALL
SELECT
    'filing_date' AS field,
    COUNT(filing_date) AS non_null_count,
    MIN(filing_date) AS minimum,
    APPROX_QUANTILE(filing_date, 0.25) AS q1,
    APPROX_QUANTILE(filing_date, 0.50) AS median,
    APPROX_QUANTILE(filing_date, 0.75) AS q3,
    MAX(filing_date) AS maximum
FROM kpss_10_25
""").df())

# Key-quality conclusion.
duplicate_patents = int(profile.loc[0, "duplicate_patent_rows"])
missing_keys = profile[["null_patent_num", "null_permno", "null_issue_date", "null_filing_date"]].iloc[0].to_dict()
print("\n=== Key-quality conclusion ===")
print("patent_num is unique:", duplicate_patents == 0)
print("Missing key counts:", missing_keys)


=== kpss_10_25 basic profile ===
 row_count  distinct_patents  duplicate_patent_rows  null_patent_num  null_permno  null_issue_date  null_filing_date min_issue_date max_issue_date min_filing_date max_filing_date
   1725791           1725791                      0                0            0                0                 0     2010-01-05     2025-12-30      1972-10-16      2025-08-21

=== Duplicate patent numbers ===
Empty DataFrame
Columns: [patent_num, occurrences, distinct_permnos]
Index: []

=== Issue-date distribution by year ===
 issue_year  patent_rows  distinct_patents
       2010        94984             94984
       2011        96464             96464
       2012       101010            101010
       2013       109181            109181
       2014       119139            119139
       2015       116483            116483
       2016       114770            114770
       2017       114738            114738
       2018       107122            107122
       2019       120640 

In [ ]:
# --- STEP 9: Check explicitly whether 2024 and 2025 data exist ---

coverage = con.execute("""
SELECT
    YEAR(issue_date) AS issue_year,
    COUNT(*) AS patent_rows,
    COUNT(DISTINCT patent_num) AS distinct_patents,
    MIN(issue_date) AS first_issue_date,
    MAX(issue_date) AS last_issue_date
FROM kpss_10_25
WHERE YEAR(issue_date) BETWEEN 2010 AND 2025
GROUP BY issue_year
ORDER BY issue_year
""").df()

print("Grant-year coverage:")
print(coverage.to_string(index=False))

for year in (2024, 2025):
    result = con.execute("""
    SELECT
        COUNT(*) AS patent_rows,
        COUNT(DISTINCT patent_num) AS distinct_patents,
        MIN(issue_date) AS first_issue_date,
        MAX(issue_date) AS last_issue_date
    FROM kpss_10_25
    WHERE YEAR(issue_date) = ?
    """, [year]).df()

    print(f"\nYear {year}: {int(result.loc[0, 'patent_rows']):,} rows")
    print(result.to_string(index=False))
